In [ ]:
# Install required dependencies
%pip install pandas numpy xgboost matplotlib gradio groq duckdb datasets python-dotenv pyarrow

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [ ]:
# ==========================================
# CELL 1: Imports & Environment Configuration
# ==========================================
import os
import json
import numpy as np
import pandas as pd
import duckdb
import xgboost as xgb
import matplotlib.pyplot as plt
import gradio as gr
from groq import Groq
from datasets import load_dataset
from dotenv import load_dotenv

# Non-interactive backend for Matplotlib inside notebooks/Gradio
plt.switch_backend('Agg')

# Load environment secrets
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Environment configured successfully!")

✅ Libraries successfully loaded!


In [ ]:
# ==========================================
# CELL 2: Cloud Dataset Loader (Hugging Face)
# ==========================================
def fetch_base_hardware_catalog() -> pd.DataFrame:
    """
    Fetches the remote Parquet hardware dataset directly from Hugging Face
    using DuckDB HTTPFS streaming.
    """
    hf_token = os.getenv("HF_TOKEN") or HF_TOKEN
    parquet_url = "https://huggingface.co/datasets/rayyanshk/dunkai/resolve/main/hardware_dataset.parquet"
    
    if not hf_token:
        raise ValueError("Missing HF_TOKEN! Please check your environment variables.")

    # Initialize DuckDB Session with direct bearer token authentication
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET extra_http_headers = {{'Authorization': 'Bearer {hf_token}'}};")

    # Fetch top candidates sample or full table
    query = f"""
        SELECT * 
        FROM '{parquet_url}' 
        LIMIT 50000
    """
    df_catalog = con.execute(query).df()
    print(f"Loaded {len(df_catalog):,} baseline hardware records from Hugging Face.")
    return df_catalog

⚡ Loaded real dataset directly from 'curated_components_dataset.csv'!
📊 Total components indexed: 616,593


In [ ]:
# ==========================================
# CELL 3: Core LTR & LLM Pipeline Logic
# ==========================================
def run_pipeline(groq_api_key, num_projects):
    api_key = groq_api_key or os.getenv("GROQ_API_KEY") or GROQ_API_KEY
    if not api_key:
        raise gr.Error("Please provide a valid Groq API Key!")

    # 1. Initialize Groq Llama Client
    client = Groq(api_key=api_key)

    prompt = f"""
    Generate {int(num_projects)} distinct electronics hardware project specifications across different industries 
    (e.g., Automotive, Medical, Robotics, Consumer IoT, Aerospace, Industrial Automation, Smart Home).

    Return ONLY a JSON object formatted strictly with a top-level key named "projects":
    {{
      "projects": [
        {{
          "project_name": "Smart Irrigation Controller",
          "domain": "Consumer IoT",
          "subsystem_label": "MCU",
          "category": "Microcontrollers (MCU/MPU/SOC)",
          "required_interfaces": ["WiFi", "I2C", "ADC"],
          "constraints": {{"min_stock": 100, "max_price_usd": 4.50}}
        }}
      ]
    }}
    """

    try:
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"}
        )
        raw_content = chat_completion.choices[0].message.content
        json_response = json.loads(raw_content)

        if "projects" in json_response:
            projects_list = json_response["projects"]
        elif isinstance(json_response, list):
            projects_list = json_response
        else:
            first_key = list(json_response.keys())[0]
            projects_list = json_response[first_key]

    except Exception as e:
        raise gr.Error(f"Groq Generation Error: {str(e)}")

    # 2. Candidate Evaluation & Synthetic Scoring Engine
    all_candidates = []
    for qid, project in enumerate(projects_list):
        req_interfaces = {i.lower() for i in project.get("required_interfaces", []) if i.lower() != "power"}
        max_price = project.get("constraints", {}).get("max_price_usd", 8.00)

        for _ in range(10):  # Evaluate 10 component candidates per query spec
            price = round(float(np.random.uniform(0.50, max_price * 1.8)), 2)
            stock = int(np.random.choice([0, np.random.randint(5, 99), np.random.randint(100, 15000)], p=[0.1, 0.2, 0.7]))
            log_stock = float(np.log1p(stock))

            match_type = np.random.choice(["perfect", "partial", "none"], p=[0.45, 0.35, 0.20])
            if match_type == "perfect":
                matched_count = len(req_interfaces)
            elif match_type == "partial" and len(req_interfaces) > 1:
                matched_count = len(req_interfaces) - 1
            else:
                matched_count = 0

            interface_score = (matched_count / len(req_interfaces)) if len(req_interfaces) > 0 else 1.0
            subcategory_match = float(np.random.choice([1.0, 0.0], p=[0.85, 0.15]))

            # Relevance Grade Decision Hierarchy {0: Bad, 1: Low Stock, 2: Over Budget, 3: Optimal}
            if matched_count < len(req_interfaces) or stock == 0 or subcategory_match == 0.0:
                relevance = 0
            elif stock < 100:
                relevance = 1
            elif price > max_price:
                relevance = 2
            else:
                relevance = 3

            all_candidates.append({
                "qid": qid,
                "project_name": project.get("project_name", f"Project #{qid}"),
                "domain": project.get("domain", "General"),
                "price": price,
                "stock": stock,
                "log_stock": log_stock,
                "interface_score": round(interface_score, 2),
                "subcategory_match": subcategory_match,
                "relevance": relevance
            })

    df_ltr = pd.DataFrame(all_candidates)

    # 3. Train XGBoost Pairwise Ranker
    feature_cols = ["price", "log_stock", "interface_score", "subcategory_match"]
    X_train = df_ltr[feature_cols].to_numpy()
    y_train = df_ltr["relevance"].to_numpy()
    qid_train = df_ltr["qid"].to_numpy()

    dtrain = xgb.DMatrix(
        X_train, 
        label=y_train, 
        qid=qid_train, 
        feature_names=feature_cols
    )

    params = {
        "objective": "rank:pairwise", 
        "learning_rate": 0.05, 
        "max_depth": 4, 
        "eval_metric": "ndcg@3"
    }
    ranker = xgb.train(params, dtrain, num_boost_round=50)

    # Predict Ranking Scores
    df_ltr["pred_score"] = ranker.predict(dtrain)
    df_ltr = df_ltr.sort_values(by=["qid", "pred_score"], ascending=[True, False])

    # 4. Metrics Formatting
    total_queries = len(projects_list)
    total_candidates_evaluated = len(df_ltr)
    optimal_matches = int((df_ltr["relevance"] == 3).sum())

    metrics_md = f"""
    ### 📊 Hardware Agent Execution Metrics
    - **Total LLM Specifications Generated:** {total_queries}
    - **Total Candidate Components Ranked:** {total_candidates_evaluated}
    - **Optimal Grade (3) Hardware Matches:** {optimal_matches}
    - **Model Objective:** `rank:pairwise` (NDCG Optimized)
    """

    # 5. Model Performance Analytics Plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Chart 1: Ground Truth Distribution
    df_ltr["relevance"].value_counts().sort_index().plot(
        kind="bar", ax=axes[0], color="#3b82f6", edgecolor="black"
    )
    axes[0].set_title("Ground Truth Relevance Grade Distribution")
    axes[0].set_xlabel("Relevance Grade (0=Bad, 3=Optimal)")
    axes[0].set_ylabel("Count")
    axes[0].grid(axis="y", linestyle="--", alpha=0.7)

    # Chart 2: Feature Importance (Gain)
    importance = ranker.get_score(importance_type="gain")
    features = list(importance.keys())
    gains = list(importance.values())

    axes[1].barh(features, gains, color="#10b981", edgecolor="black")
    axes[1].set_title("XGBoost Feature Importances (Gain)")
    axes[1].set_xlabel("Total Gain")
    axes[1].grid(axis="x", linestyle="--", alpha=0.7)

    plt.tight_layout()

    return metrics_md, df_ltr, fig

In [ ]:
# ==========================================
# CELL 4: Gradio UI Interface Block
# ==========================================
def build_dashboard():
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# ⚡ Hardware Recommendation Agent (Groq + DuckDB + XGBoost)")
        gr.Markdown("An end-to-end Learning-to-Rank agent streaming cloud dataset catalogs, creating LLM project specs, and ranking component candidates.")

        with gr.Row():
            api_key_input = gr.Textbox(
                label="Groq API Key", 
                type="password", 
                placeholder="gsk_...", 
                value=os.environ.get("GROQ_API_KEY", "")
            )
            num_queries_slider = gr.Slider(
                minimum=5, maximum=30, value=15, step=1, 
                label="Number of Groq Specs to Generate"
            )

        run_button = gr.Button("🚀 Execute Pipeline & Train Ranker", variant="primary")

        with gr.Row():
            metrics_output = gr.Markdown(label="Pipeline Metrics")

        with gr.Row():
            plot_output = gr.Plot(label="Model Feature Analytics")

        with gr.Row():
            dataframe_output = gr.Dataframe(label="Ranked Candidate Candidates Output", wrap=True)

        run_button.click(
            fn=run_pipeline,
            inputs=[api_key_input, num_queries_slider],
            outputs=[metrics_output, dataframe_output, plot_output]
        )
    return demo

# Initialize application
app = build_dashboard()

In [ ]:
# ==========================================
# CELL 5: Launch Dashboard
# ==========================================
if __name__ == "__main__":
    app.launch(inline=True)

🚀 Agent #3 Execution Completed in 2291.20 ms!

{
  "project_name": "Smart Weather System",
  "total_bom_cost_usd": 5.35,
  "total_components": 9,
  "bill_of_materials": [
    {
      "subsystem_id": "mcu",
      "label": "MCU",
      "selected_part": {
        "lcsc_part": "C82891",
        "mfr_part": "ESP-12F(ESP8266MOD)",
        "manufacturer": "Ai-Thinker",
        "package": "SMD,24x16mm",
        "unit_price_usd": 2.237,
        "in_stock_qty": 13728,
        "description": "-40\u2103~+85\u2103 -90dBm 120mA 16dBm 170mA 2.4GHz 3.3V ESP8266 Chip On-board PCB Antenna UART\u3001SPI\u3001I2C SMD,24x16mm WiFi Modules ROHS"
      },
      "selection_rationale": "Ranked #1 by XGBoost LTR. Stock: 13728 units | Interfaces: ['I2C', 'SPI', 'WiFi', 'Power']"
    },
    {
      "subsystem_id": "battery",
      "label": "Battery",
      "selected_part": {
        "lcsc_part": "C7440",
        "mfr_part": "PCF8563T/5,518",
        "manufacturer": "NXP Semicon",
        "package": "SO-8",
      